# Display WebGL Viewer in a Jupyter Notebook

Pycortex can embed the interactive WebGL brain viewer directly in a Jupyter
notebook cell. There are two methods:

1. **IFrame mode** (default): Starts a background Tornado server and embeds
   it in an IFrame. Provides full interactivity including surface morphing,
   data switching, and programmatic control from Python via WebSocket.

2. **Static mode**: Generates a self-contained HTML viewer that can be
   embedded inline. Works in static notebook renderers like nbviewer.

Both methods are **non-blocking** — subsequent notebook cells execute immediately.

## Create some example data

In [ ]:
import cortex
import numpy as np

np.random.seed(1234)
volume = cortex.Volume.random(subject='S1', xfmname='fullhead')

## Static viewer

`make_notebook_html()` generates a self-contained HTML string with the
WebGL viewer and all data embedded. Wrapping it in an IFrame via
`IPython.display.HTML` renders the interactive 3-D viewer inline.

In [ ]:
from IPython.display import HTML

html = cortex.webgl.jupyter.make_notebook_html(volume)
HTML(
    '<iframe srcdoc="{srcdoc}" '
    'style="border:none; width:100%; height:600px;">'
    '</iframe>'.format(srcdoc=html.replace('"', '&quot;'))
)

## IFrame mode (for live Jupyter sessions)

When running in a live Jupyter session, the IFrame mode starts a Tornado
server in a background thread and embeds the viewer in an IFrame. This
provides full interactivity including surface morphing, data switching,
and programmatic control from Python via WebSocket.

```python
# Start the viewer (non-blocking)
server = cortex.webgl.jupyter.display(volume)

# In a subsequent cell, get a control handle
client = server.get_client()

# Rotate the view
client._set_view(azimuth=45, altitude=30)

# Switch to inflated surface (mix=1.0 is fully inflated)
client._set_view(mix=1.0)

# Capture a screenshot
client.getImage("notebook_screenshot.png")
```

> **Note:** `get_client()` blocks until the browser connects via WebSocket,
> so call it in a **separate cell** from `display()`.

## Convenience wrapper: `display(method="static")`

For quick use in a live notebook, you can also call `display()` with
`method="static"` which generates the static viewer and serves it via a
lightweight local HTTP server:

```python
cortex.webgl.jupyter.display(volume, method="static")
```

## Customizing the viewer

Both methods accept the same keyword arguments as `cortex.webgl.show()`
and `cortex.webgl.make_static()`. For example:

```python
cortex.webgl.jupyter.display(
    volume,
    types=("inflated",),            # Surface types to include
    overlays_visible=("sulci",),    # Show sulci overlay by default
    height=400,                      # Shorter viewer
    title="My Experiment",
)
```